In [3]:
import pandas as pd
import numpy as np
import itertools
import pymc as pm
import arviz as az

import sys
sys.path.append('../Source')
from Pipeline.build_country_comparisons import buildCountryComparisons
from Pipeline.uncertainty_metrics import computeUncertainty

alignedCountryData = pd.read_csv('../Data/Processed/alignedDataCountry.csv')
countryComparisons = buildCountryComparisons(alignedCountryData)

print(countryComparisons.shape)

c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\arviz\__init__.py:50: FutureWarning: 
ArviZ is undergoing a major refactor to improve flexibility and extensibility while maintaining a user-friendly interface.
Some upcoming changes may be backward incompatible.
For details and migration guidance, visit: https://python.arviz.org/en/latest/user_guide/migration_guide.html
  warn(


(232525, 6)


In [4]:
caucasusData = countryComparisons[countryComparisons['region'] == 'Caucasus and Central Asia']

caucasusWeeks = sorted(caucasusData['week'].unique())
print(len(caucasusWeeks))
print(caucasusWeeks[:5])

391
[Period('2018-01-01/2018-01-07', 'W-SUN'), Period('2018-01-08/2018-01-14', 'W-SUN'), Period('2018-01-15/2018-01-21', 'W-SUN'), Period('2018-01-22/2018-01-28', 'W-SUN'), Period('2018-01-29/2018-02-04', 'W-SUN')]


In [5]:
regionCountries = sorted(set(caucasusData['countryI']) | set(caucasusData['countryJ']))
regionIndexMap = {country: i for i, country in enumerate(regionCountries)}

nRegionCountries = len(regionCountries)
print(nRegionCountries)
print(regionCountries)

9
['Afghanistan', 'Armenia', 'Azerbaijan', 'Georgia', 'Kazakhstan', 'Kyrgyzstan', 'Tajikistan', 'Turkmenistan', 'Uzbekistan']


In [2]:
initialWeekData = caucasusData[caucasusData['week'] == caucasusWeeks[0]]

initialIndexI = initialWeekData['countryI'].map(regionIndexMap).to_numpy()
initialIndexJ = initialWeekData['countryJ'].map(regionIndexMap).to_numpy()
initialWinsI = initialWeekData['winsI'].to_numpy()
initialTotals  = initialWeekData['total'].to_numpy()

with pm.Model() as rollingModel:
    indexIData = pm.Data('indexIData', initialIndexI)
    indexJData = pm.Data('indexJData', initialIndexJ)
    winsIData = pm.Data('winsIData', initialWinsI)
    totalsData = pm.Data('totalsData', initialTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nRegionCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

NameError: name 'caucasusData' is not defined

In [ ]:
# NUTS Approach

import time

weeklyResults = []
weeksSkipped = []

startTime = time.time()

for week in caucasusWeeks:
    weekData = caucasusData[caucasusData['week'] == week]

    if len(weekData) == 0:
        weeksSkipped.append(week)
        continue

    indexI = weekData['countryI'].map(regionIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(regionIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        trace = pm.sample(200, tune=200, target_accept=0.9, return_inferencedata=True, chains=2, cores=2, progressbar=False)

    weeklyResults.append({'week': week, 'trace': trace})

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(weeklyResults)}")
print(f"Weeks skipped: {len(weeksSkipped)}")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 200 tune and 200 draw iterations (400 + 400 draws total) took 14 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 200 tune and 200 draw iterations (400 + 400 draws total) took 15 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher

KeyboardInterrupt: 

In [ ]:
# ADVI approach

import time 

weeklyMetrics = []
weeksSkipped = []

startTime = time.time()

for week in caucasusWeeks:
    weekData = caucasusData[caucasusData['week'] == week]

    if len(weekData) == 0:
        weeksSkipped.append(week)
        continue

    indexI = weekData['countryI'].map(regionIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(regionIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    results = computeUncertainty(trace, regionCountries)

    weeklyMetrics.append({
        'week': week,
        'region': 'Caucasus and Central Asia',
        'entropy': results['entropy'],
        'flipProbabilities': results['flipProbabilities'],
        'hdiOverlaps': results['hdiOverlaps']
    })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(weeklyMetrics)}")
print(f"Weeks skipped: {len(weeksSkipped)}")

Finished [100%]: Average Loss = 48.595
Finished [100%]: Average Loss = 61.539
Finished [100%]: Average Loss = 48.663
Finished [100%]: Average Loss = 61.672
Finished [100%]: Average Loss = 60.882
Finished [100%]: Average Loss = 63.613
Finished [100%]: Average Loss = 49.456
Finished [100%]: Average Loss = 49.96
Finished [100%]: Average Loss = 50.164
Finished [100%]: Average Loss = 72.561
Finished [100%]: Average Loss = 60.577
Finished [100%]: Average Loss = 60.089
Finished [100%]: Average Loss = 59.456
Finished [100%]: Average Loss = 75.127
Finished [100%]: Average Loss = 63.773
Finished [100%]: Average Loss = 51.667
Finished [100%]: Average Loss = 64.329
Finished [100%]: Average Loss = 73.337
Finished [100%]: Average Loss = 65.097
Finished [100%]: Average Loss = 51.327
Finished [100%]: Average Loss = 61.283
Finished [100%]: Average Loss = 64.072
Finished [100%]: Average Loss = 76.527
Finished [100%]: Average Loss = 60.577
Finished [100%]: Average Loss = 50.696
Finished [100%]: Average L

Total time: 961.0 seconds
Weeks fit: 391
Weeks skipped: 0


In [ ]:
weeklyMetricsDF = pd.DataFrame(weeklyMetrics)
print(weeklyMetricsDF[['week', 'entropy']].describe())
print(weeklyMetricsDF[['week', 'entropy']].sort_values('entropy', ascending=False).head(10))

          entropy
count  391.000000
mean     4.256846
std      1.168268
min      1.135404
25%      3.393984
50%      4.373560
75%      5.335824
max      5.942944
                      week   entropy
390  2025-06-23/2025-06-29  5.942944
388  2025-06-09/2025-06-15  5.905671
304  2023-10-30/2023-11-05  5.897890
330  2024-04-29/2024-05-05  5.877945
248  2022-10-03/2022-10-09  5.866239
317  2024-01-29/2024-02-04  5.859308
292  2023-08-07/2023-08-13  5.853226
368  2025-01-20/2025-01-26  5.849760
306  2023-11-13/2023-11-19  5.837205
350  2024-09-16/2024-09-22  5.833740


In [ ]:
weeklyMetricsDF.to_csv('../Data/Processed/weeklyUncertaintyCaucasusCA_raw.csv', index=False)
print(weeklyMetricsDF.shape)

(391, 5)


In [ ]:
weeklyMetricsDF = pd.read_csv('../Data/Processed/weeklyUncertaintyCaucasusCA_raw.csv')

lowestEntropyWeek = weeklyMetricsDF.sort_values('entropy').iloc[0]['week']
highEntropyWeek1 = '2025-06-23/2025-06-29'
highEntropyWeek2 = '2023-10-30/2023-11-05'

testWeeksValidation = [lowestEntropyWeek, highEntropyWeek1, highEntropyWeek2]
print(testWeeksValidation)

['2020-06-15/2020-06-21', '2025-06-23/2025-06-29', '2023-10-30/2023-11-05']


In [ ]:
nutsValidation = []

for week in testWeeksValidation:
    weekPeriod = pd.Period(week, freq='W-SUN')
    weekData = caucasusData[caucasusData['week'] == weekPeriod]

    indexI = weekData['countryI'].map(regionIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(regionIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModel:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        trace = pm.sample(500, tune=500, target_accept=0.9, return_inferencedata=True, chains=2, cores=2, progressbar=False)

    results = computeUncertainty(trace, regionCountries)
    nutsValidation.append({'week': week, 'entropy': results['entropy']})

    print(f"{week}: NUTS entropy = {results['entropy']:.3f}")

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 10 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
Initializing NUTS using jitter+adapt_diag...


2020-06-15/2020-06-21: NUTS entropy = 1.000


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 11 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
Initializing NUTS using jitter+adapt_diag...


2025-06-23/2025-06-29: NUTS entropy = 6.802


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 10 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


2023-10-30/2023-11-05: NUTS entropy = 6.610


In [ ]:
comparison = weeklyMetricsDF[weeklyMetricsDF['week'].isin(testWeeksValidation)][['week', 'entropy']]
comparison = comparison.rename(columns={'entropy': 'adviEntropy'})

nutsDF = pd.DataFrame(nutsValidation).rename(columns={'entropy': 'nutsEntropy'})
comparison = comparison.merge(nutsDF, on='week')

print(comparison)

                    week  adviEntropy  nutsEntropy
0  2020-06-15/2020-06-21     1.135404     0.999642
1  2023-10-30/2023-11-05     5.897890     6.609835
2  2025-06-23/2025-06-29     5.942944     6.802213


In [ ]:
weeklyMetricsDF.head()

,week,region,entropy,flipProbabilities,hdiOverlaps
0,2018-01-01/2018-01-07,Caucasus and Central Asia,4.175930,"{'Afghanistan over Armenia': np.float64(1.0), ...","{'Afghanistan vs Armenia': np.False_, 'Afghani..."
1,2018-01-08/2018-01-14,Caucasus and Central Asia,3.396296,"{'Afghanistan over Armenia': np.float64(1.0), ...","{'Afghanistan vs Armenia': np.False_, 'Afghani..."
2,2018-01-15/2018-01-21,Caucasus and Central Asia,3.535702,"{'Afghanistan over Armenia': np.float64(1.0), ...","{'Afghanistan vs Armenia': np.False_, 'Afghani..."
3,2018-01-22/2018-01-28,Caucasus and Central Asia,2.841213,"{'Afghanistan over Armenia': np.float64(1.0), ...","{'Afghanistan vs Armenia': np.False_, 'Afghani..."
4,2018-01-29/2018-02-04,Caucasus and Central Asia,3.116704,"{'Afghanistan over Armenia': np.float64(1.0), ...","{'Afghanistan vs Armenia': np.False_, 'Afghani..."


In [16]:
import re

def parseMetricDict(value):
    if not isinstance(value, str):
        return value
    cleaned = re.sub(r'np\.float64\(([^)]+)\)', r'\1', value)
    cleaned = cleaned.replace('np.False_', 'False').replace('np.True_', 'True')
    return ast.literal_eval(cleaned)

In [ ]:
import ast 

longData = []

for _, row in weeklyMetricsDF.iterrows(): 
    week = row['week']
    region = row['region']
    entropy = row['entropy']

    flipProbs = parseMetricDict(row['flipProbabilities'])
    hdiOverlaps = parseMetricDict(row['hdiOverlaps'])

    for pairLabel, flipProb in flipProbs.items(): 
        countryI, countryJ = pairLabel.split(' over ')
        overlap = hdiOverlaps.get(f'{countryI} vs {countryJ}', None)

        longData.append({
            'week': week,
            'region': region,
            'entropy': entropy,
            'countryI': countryI,
            'countryJ': countryJ,
            'flipProbability': flipProb,
            'hdiOverlap': overlap
        })

longFormDF = pd.DataFrame(longData)
print(longFormDF.shape)
longFormDF.head()

(14076, 7)


,week,region,entropy,countryI,countryJ,flipProbability,hdiOverlap
0,2018-01-01/2018-01-07,Caucasus and Central Asia,4.17593,Afghanistan,Armenia,1.00,False
1,2018-01-01/2018-01-07,Caucasus and Central Asia,4.17593,Afghanistan,Azerbaijan,0.58,True
2,2018-01-01/2018-01-07,Caucasus and Central Asia,4.17593,Afghanistan,Georgia,1.00,False
3,2018-01-01/2018-01-07,Caucasus and Central Asia,4.17593,Afghanistan,Kazakhstan,1.00,False
4,2018-01-01/2018-01-07,Caucasus and Central Asia,4.17593,Afghanistan,Kyrgyzstan,1.00,False


In [ ]:
longFormDF.to_csv('../Data/Processed/weeklyUncertaintyCaucasusCA.csv', index=False)
print(longFormDF.shape)

(14076, 7)


In [ ]:
middleEastData = countryComparisons[countryComparisons['region'] == 'Middle East']

middleEastCountries = sorted(set(middleEastData['countryI']) | set(middleEastData['countryJ']))
middleEastIndexMap = {country: i for i, country in enumerate(middleEastCountries)}
nMiddleEastCountries = len(middleEastCountries)

middleEastWeeks = sorted(middleEastData['week'].unique())

print(nMiddleEastCountries)
print(len(middleEastWeeks))

13
391


In [ ]:
initialWeekData = middleEastData[middleEastData['week'] == middleEastWeeks[0]]

initialIndexI = initialWeekData['countryI'].map(middleEastIndexMap).to_numpy()
initialIndexJ = initialWeekData['countryJ'].map(middleEastIndexMap).to_numpy()
initialWinsI = initialWeekData['winsI'].to_numpy()
initialTotals  = initialWeekData['total'].to_numpy()

with pm.Model() as rollingModelME:
    indexIData = pm.Data('indexIData', initialIndexI)
    indexJData = pm.Data('indexJData', initialIndexJ)
    winsIData = pm.Data('winsIData', initialWinsI)
    totalsData = pm.Data('totalsData', initialTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nMiddleEastCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [ ]:
import time

weeklyMetricsME = []
weeksSkippedME = []

startTime = time.time()

for week in middleEastWeeks:
    weekData = middleEastData[middleEastData['week'] == week]

    if len(weekData) == 0:
        weeksSkippedME.append(week)
        continue

    indexI = weekData['countryI'].map(middleEastIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(middleEastIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModelME:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    results = computeUncertainty(trace, middleEastCountries)

    weeklyMetricsME.append({
        'week': week,
        'region': 'Middle East',
        'entropy': results['entropy'],
        'flipProbabilities': results['flipProbabilities'],
        'hdiOverlaps': results['hdiOverlaps']
    })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(weeklyMetricsME)}")
print(f"Weeks skipped: {len(weeksSkippedME)}")

Finished [100%]: Average Loss = 138.68
Finished [100%]: Average Loss = 168.35
Finished [100%]: Average Loss = 170.04
Finished [100%]: Average Loss = 171.02
Finished [100%]: Average Loss = 175.42
Finished [100%]: Average Loss = 174.92
Finished [100%]: Average Loss = 173.48
Finished [100%]: Average Loss = 169.81
Finished [100%]: Average Loss = 162.58
Finished [100%]: Average Loss = 170.7
Finished [100%]: Average Loss = 173.92
Finished [100%]: Average Loss = 167.65
Finished [100%]: Average Loss = 157.88
Finished [100%]: Average Loss = 178.31
Finished [100%]: Average Loss = 173.13
Finished [100%]: Average Loss = 168.89
Finished [100%]: Average Loss = 167.95
Finished [100%]: Average Loss = 165.38
Finished [100%]: Average Loss = 188.79
Finished [100%]: Average Loss = 197.09
Finished [100%]: Average Loss = 167.81
Finished [100%]: Average Loss = 171.8
Finished [100%]: Average Loss = 169.02
Finished [100%]: Average Loss = 162.18
Finished [100%]: Average Loss = 161.48
Finished [100%]: Average Lo

Total time: 1020.5 seconds
Weeks fit: 391
Weeks skipped: 0


In [ ]:
weeklyMetricsMEDF = pd.DataFrame(weeklyMetricsME)
print(weeklyMetricsMEDF[['week', 'entropy']].describe())

          entropy
count  391.000000
mean     3.778652
std      1.011246
min      0.800093
25%      3.092180
50%      3.823763
75%      4.543043
max      5.927773


In [ ]:
longRowsME = []

for _, row in weeklyMetricsMEDF.iterrows():
    week = row['week']
    region = row['region']
    entropy = row['entropy']

    flipProbs = parseMetricDict(row['flipProbabilities'])
    hdiOverlaps = parseMetricDict(row['hdiOverlaps'])

    for pairLabel, flipProb in flipProbs.items():
        countryI, countryJ = pairLabel.split(' over ')
        overlap = hdiOverlaps.get(f"{countryI} vs {countryJ}", None)

        longRowsME.append({
            'week': week,
            'region': region,
            'entropy': entropy,
            'countryI': countryI,
            'countryJ': countryJ,
            'flipProbability': flipProb,
            'hdiOverlap': overlap
        })

longFormMEDF = pd.DataFrame(longRowsME)
print(longFormMEDF.shape)
longFormMEDF.head()

(30498, 7)


,week,region,entropy,countryI,countryJ,flipProbability,hdiOverlap
0,2018-01-01/2018-01-07,Middle East,4.774322,Bahrain,Iran,0.0000,False
1,2018-01-01/2018-01-07,Middle East,4.774322,Bahrain,Iraq,0.9725,True
2,2018-01-01/2018-01-07,Middle East,4.774322,Bahrain,Israel,1.0000,False
3,2018-01-01/2018-01-07,Middle East,4.774322,Bahrain,Jordan,1.0000,False
4,2018-01-01/2018-01-07,Middle East,4.774322,Bahrain,Kuwait,0.1900,True


In [ ]:
longFormMEDF.to_csv('../Data/Processed/weeklyUncertaintyMiddleEast.csv', index=False)

In [ ]:
northAfricaData = countryComparisons[countryComparisons['region'] == 'Northern Africa']

northAfricaCountries = sorted(set(northAfricaData['countryI']) | set(northAfricaData['countryJ']))
northAfricaIndexMap = {country: i for i, country in enumerate(northAfricaCountries)}
nNorthAfricaCountries = len(northAfricaCountries)

northAfricaWeeks = sorted(northAfricaData['week'].unique())

print(nNorthAfricaCountries)
print(len(northAfricaWeeks))

6
391


In [ ]:
initialWeekData = northAfricaData[northAfricaData['week'] == northAfricaWeeks[0]]

initialIndexI = initialWeekData['countryI'].map(northAfricaIndexMap).to_numpy()
initialIndexJ = initialWeekData['countryJ'].map(northAfricaIndexMap).to_numpy()
initialWinsI = initialWeekData['winsI'].to_numpy()
initialTotals  = initialWeekData['total'].to_numpy()

with pm.Model() as rollingModelNA:
    indexIData = pm.Data('indexIData', initialIndexI)
    indexJData = pm.Data('indexJData', initialIndexJ)
    winsIData = pm.Data('winsIData', initialWinsI)
    totalsData = pm.Data('totalsData', initialTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nNorthAfricaCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [ ]:
weeklyMetricsNA = []
weeksSkippedNA = []

startTime = time.time() 

for week in northAfricaWeeks: 
    weekData = northAfricaData[northAfricaData['week'] == week]

    if len(weekData) == 0: 
        weeksSkippedNA.append(week)
        continue

    indexI = weekData['countryI'].map(northAfricaIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(northAfricaIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModelNA: 
        pm.set_data({
            'indexIData' : indexI, 
            'indexJData' : indexJ, 
            'winsIData' : winsI, 
            'totalsData' : totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    results = computeUncertainty(trace, northAfricaCountries)

    weeklyMetricsNA.append({
        'week': week,
        'region': 'Northern Africa',
        'entropy': results['entropy'],
        'flipProbabilities': results['flipProbabilities'],
        'hdiOverlaps': results['hdiOverlaps']
    })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(weeklyMetricsNA)}")
print(f"Weeks skipped: {len(weeksSkippedNA)}")

Finished [100%]: Average Loss = 32.269
Finished [100%]: Average Loss = 44.187
Finished [100%]: Average Loss = 38.215
Finished [100%]: Average Loss = 34.988
Finished [100%]: Average Loss = 35.894
Finished [100%]: Average Loss = 31.552
Finished [100%]: Average Loss = 34.709
Finished [100%]: Average Loss = 33.673
Finished [100%]: Average Loss = 32.439
Finished [100%]: Average Loss = 34.466
Finished [100%]: Average Loss = 33.092
Finished [100%]: Average Loss = 32.531
Finished [100%]: Average Loss = 23.198
Finished [100%]: Average Loss = 16.658
Finished [100%]: Average Loss = 29.653
Finished [100%]: Average Loss = 30.99
Finished [100%]: Average Loss = 32.309
Finished [100%]: Average Loss = 28.303
Finished [100%]: Average Loss = 29.213
Finished [100%]: Average Loss = 32.329
Finished [100%]: Average Loss = 34.112
Finished [100%]: Average Loss = 25.902
Finished [100%]: Average Loss = 31.787
Finished [100%]: Average Loss = 23.565
Finished [100%]: Average Loss = 28.875
Finished [100%]: Average L

Total time: 771.3 seconds
Weeks fit: 391
Weeks skipped: 0


In [ ]:
weeklyMetricsNADF = pd.DataFrame(weeklyMetricsNA)
print(weeklyMetricsNADF[['week', 'entropy']].describe())

          entropy
count  391.000000
mean     1.631581
std      0.833683
min      0.139355
25%      1.032420
50%      1.494632
75%      2.097244
max      5.164969


In [ ]:
longRowsNA = []

for _, row in weeklyMetricsNADF.iterrows():
    week = row['week']
    region = row['region']
    entropy = row['entropy']

    flipProbs = parseMetricDict(row['flipProbabilities'])
    hdiOverlaps = parseMetricDict(row['hdiOverlaps'])

    for pairLabel, flipProb in flipProbs.items():
        countryI, countryJ = pairLabel.split(' over ')
        overlap = hdiOverlaps.get(f"{countryI} vs {countryJ}", None)

        longRowsNA.append({
            'week': week,
            'region': region,
            'entropy': entropy,
            'countryI': countryI,
            'countryJ': countryJ,
            'flipProbability': flipProb,
            'hdiOverlap': overlap
        })

longFormNADF = pd.DataFrame(longRowsNA)
print(longFormNADF.shape)

(5865, 7)


In [ ]:
longFormNADF.to_csv('../Data/Processed/weeklyUncertaintyNorthAfrica.csv', index=False)

In [6]:
europeData = countryComparisons[countryComparisons['region'] == 'Europe']

europeCountries = sorted(set(europeData['countryI']) | set(europeData['countryJ']))
europeIndexMap = {country: i for i, country in enumerate(europeCountries)}
nEuropeCountries = len(europeCountries)

europeWeeks = sorted(europeData['week'].unique())

print(nEuropeCountries)
print(len(europeWeeks))

41
391


In [10]:
initialWeekData = europeData[europeData['week'] == europeWeeks[0]]

initialIndexI = initialWeekData['countryI'].map(europeIndexMap).to_numpy()
initialIndexJ = initialWeekData['countryJ'].map(europeIndexMap).to_numpy()
initialWinsI = initialWeekData['winsI'].to_numpy()
initialTotals  = initialWeekData['total'].to_numpy()

with pm.Model() as rollingModelEurope:
    indexIData = pm.Data('indexIData', initialIndexI)
    indexJData = pm.Data('indexJData', initialIndexJ)
    winsIData = pm.Data('winsIData', initialWinsI)
    totalsData = pm.Data('totalsData', initialTotals)

    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nEuropeCountries)
    p = pm.math.sigmoid(beta[indexIData] - beta[indexJData])
    obs = pm.Binomial('obs', n=totalsData, p=p, observed=winsIData)

In [11]:
import time

weeklyMetricsEurope = []
weeksSkippedEurope = []

startTime = time.time()

for week in europeWeeks:
    weekData = europeData[europeData['week'] == week]

    if len(weekData) == 0:
        weeksSkippedEurope.append(week)
        continue

    indexI = weekData['countryI'].map(europeIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(europeIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModelEurope:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    results = computeUncertainty(trace, europeCountries)

    weeklyMetricsEurope.append({
        'week': week,
        'region': 'Europe',
        'entropy': results['entropy'],
        'flipProbabilities': results['flipProbabilities'],
        'hdiOverlaps': results['hdiOverlaps']
    })

endTime = time.time()

print(f"Total time: {endTime - startTime:.1f} seconds")
print(f"Weeks fit: {len(weeklyMetricsEurope)}")
print(f"Weeks skipped: {len(weeksSkippedEurope)}")

Finished [100%]: Average Loss = 48.034
Finished [100%]: Average Loss = 100.1
Finished [100%]: Average Loss = 144.49
Finished [100%]: Average Loss = 134.44
Finished [100%]: Average Loss = 163.07
Finished [100%]: Average Loss = 151.21
Finished [100%]: Average Loss = 156.72
Finished [100%]: Average Loss = 164.45
Finished [100%]: Average Loss = 179.03
Finished [100%]: Average Loss = 177.02
Finished [100%]: Average Loss = 145.39
Finished [100%]: Average Loss = 138.01
Finished [100%]: Average Loss = 142.12
Finished [100%]: Average Loss = 168.66
Finished [100%]: Average Loss = 104.91
Finished [100%]: Average Loss = 154.81
Finished [100%]: Average Loss = 159.61
Finished [100%]: Average Loss = 147.57
Finished [100%]: Average Loss = 137.78
Finished [100%]: Average Loss = 158.6
Finished [100%]: Average Loss = 159.2
Finished [100%]: Average Loss = 131.94
Finished [100%]: Average Loss = 199.94
Finished [100%]: Average Loss = 198.29
Finished [100%]: Average Loss = 196.06
Finished [100%]: Average Los

Total time: 1042.2 seconds
Weeks fit: 391
Weeks skipped: 0


In [12]:
weeklyMetricsEuropeDF = pd.DataFrame(weeklyMetricsEurope)
print(weeklyMetricsEuropeDF[['week', 'entropy']].describe())

          entropy
count  391.000000
mean     5.991465
std      0.000000
min      5.991465
25%      5.991465
50%      5.991465
75%      5.991465
max      5.991465


In [13]:
sampleWeek1 = weeklyMetricsEurope[0]
sampleWeek2 = weeklyMetricsEurope[200]

print(sampleWeek1['week'], sampleWeek1['entropy'])
print(sampleWeek2['week'], sampleWeek2['entropy'])
print(sampleWeek1['flipProbabilities'] == sampleWeek2['flipProbabilities'])

2018-01-01/2018-01-07 5.991464547107983
2021-11-01/2021-11-07 5.991464547107983
False


In [14]:
import numpy as np
from scipy.stats import entropy as entropyFunc

def diagnosticEntropy(trace, nCountries): 
    samples = trace.posterior['beta'].values.reshape(-1, nCountries)
    rankings = np.argsort(-samples, axis=1)
    uniqueRankings, counts = np.unique(rankings, axis=0, return_counts=True)
    probabilities = counts /counts.sum()
    return len(uniqueRankings), counts.max(), entropyFunc(probabilities)                                                    

diagnosticWeeks = ['2018-01-01/2018-01-07', '2021-11-01/2021-11-07']

for week in diagnosticWeeks:
    weekPeriod = pd.Period(week, freq='W-SUN')
    weekData = europeData[europeData['week'] == weekPeriod]

    indexI = weekData['countryI'].map(europeIndexMap).to_numpy()
    indexJ = weekData['countryJ'].map(europeIndexMap).to_numpy()
    winsI = weekData['winsI'].to_numpy()
    totals = weekData['total'].to_numpy()

    with rollingModelEurope:
        pm.set_data({
            'indexIData': indexI,
            'indexJData': indexJ,
            'winsIData': winsI,
            'totalsData': totals
        })

        approx = pm.fit(n=20000, method='advi', progressbar=False)
        trace = approx.sample(400)

    nUnique, maxCount, ent = diagnosticEntropy(trace, nEuropeCountries)
    print(f"{week}: unique rankings = {nUnique} out of 400 draws, max single-ranking count = {maxCount}, entropy = {ent:.4f}")

Finished [100%]: Average Loss = 47.995


2018-01-01/2018-01-07: unique rankings = 400 out of 400 draws, max single-ranking count = 1, entropy = 5.9915


Finished [100%]: Average Loss = 1,049.9


2021-11-01/2021-11-07: unique rankings = 400 out of 400 draws, max single-ranking count = 1, entropy = 5.9915


In [17]:
longRowsEurope = []

for _, row in weeklyMetricsEuropeDF.iterrows():
    week = row['week']
    region = row['region']
    entropy = row['entropy']

    flipProbs = parseMetricDict(row['flipProbabilities'])
    hdiOverlaps = parseMetricDict(row['hdiOverlaps'])

    for pairLabel, flipProb in flipProbs.items():
        countryI, countryJ = pairLabel.split(' over ')
        overlap = hdiOverlaps.get(f"{countryI} vs {countryJ}", None)

        longRowsEurope.append({
            'week': week,
            'region': region,
            'entropy': entropy,
            'entropyReliable': False,
            'countryI': countryI,
            'countryJ': countryJ,
            'flipProbability': flipProb,
            'hdiOverlap': overlap
        })

longFormEuropeDF = pd.DataFrame(longRowsEurope)
print(longFormEuropeDF.shape)

(320620, 8)


In [18]:
longFormEuropeDF.to_csv('../Data/Processed/weeklyUncertaintyEurope.csv', index=False)